# NHL Data Cleaning

This notebook cleans and preprocesses the raw NHL play-by-play data.

## Steps:
1. Filter for shot events only
2. Flatten nested JSON columns
3. Remove unnecessary columns
4. Standardize coordinates
5. Handle missing data
6. Create target variable (is_goal)


In [1]:
import pandas as pd
import numpy as np
import sys

# Add src to path
sys.path.append('../../')
from src.data.processors.data_cleaning import (
    filter_shot_events,
    flatten_nested_columns,
    remove_unnecessary_columns,
    standardize_coordinates,
    handle_missing_data,
    create_goal_target
)


## Load Raw Data


In [2]:
# Load raw data
INPUT_FILE = "../../data/raw/nhl_raw_plays.parquet"
df = pd.read_parquet(INPUT_FILE)

print(f"Original shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


Original shape: (1820932, 11)
Columns: ['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder', 'details', 'pptReplayUrl']


,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl
0,8,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:00,20:00,1551,right,520,period-start,8,None,None
1,9,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:00,20:00,1551,right,502,faceoff,10,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
2,10,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:25,19:35,1551,right,505,goal,11,"{'assist1PlayerId': 8477015.0, 'assist1PlayerT...",None
3,11,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:25,19:35,1551,right,502,faceoff,14,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
4,12,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:38,19:22,1551,right,507,missed-shot,15,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None


## Step 1: Filter for Shot Events


In [3]:
# Identify all unique events
print("Unique event types:")
print(df["typeDescKey"].unique())

# Filter for shot events
shot_events = ['shot-on-goal', 'missed-shot', 'blocked-shot', 'goal']
df_shots = filter_shot_events(df, shot_events)

print(f"\nOriginal shape: {df.shape}")
print(f"After filtering shots: {df_shots.shape}")
df_shots.head()


Unique event types:
['period-start' 'faceoff' 'goal' 'missed-shot' 'stoppage'
 'delayed-penalty' 'penalty' 'giveaway' 'shot-on-goal' 'blocked-shot'
 'hit' 'takeaway' 'period-end' 'game-end' 'shootout-complete'
 'failed-shot-attempt']

Original shape: (1820932, 11)
After filtering shots: (680581, 11)


,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl
2,10,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:25,19:35,1551,right,505,goal,11,"{'assist1PlayerId': 8477015.0, 'assist1PlayerT...",None
4,12,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:38,19:22,1551,right,507,missed-shot,15,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
11,15,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:31,18:29,1451,right,506,shot-on-goal,28,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
14,50,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:39,18:21,1451,right,508,blocked-shot,32,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
15,18,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:58,18:02,1451,right,507,missed-shot,33,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None


## Step 2: Flatten Nested Columns


In [4]:
# Flatten nested JSON columns
df_shots_clean = flatten_nested_columns(df_shots)

print(f"Shape after flattening: {df_shots_clean.shape}")
print(f"\nColumns ({len(df_shots_clean.columns)}):")
for col in df_shots_clean.columns:
    print(f"  - {col}")
df_shots_clean.head()


Removed duplicate columns. New shape: (680581, 48)
Shape after flattening: (680581, 48)

Columns (48):
  - eventId
  - timeInPeriod
  - timeRemaining
  - situationCode
  - homeTeamDefendingSide
  - typeCode
  - typeDescKey
  - sortOrder
  - pptReplayUrl
  - assist1PlayerId
  - assist1PlayerTotal
  - assist2PlayerId
  - assist2PlayerTotal
  - awaySOG
  - awayScore
  - blockingPlayerId
  - committedByPlayerId
  - descKey
  - discreteClip
  - discreteClipFr
  - drawnByPlayerId
  - duration
  - eventOwnerTeamId
  - goalieInNetId
  - highlightClip
  - highlightClipFr
  - highlightClipSharingUrl
  - highlightClipSharingUrlFr
  - hitteePlayerId
  - hittingPlayerId
  - homeSOG
  - homeScore
  - losingPlayerId
  - playerId
  - reason
  - scoringPlayerId
  - scoringPlayerTotal
  - secondaryReason
  - servedByPlayerId
  - shootingPlayerId
  - shotType
  - winningPlayerId
  - xCoord
  - yCoord
  - zoneCode
  - maxRegulationPeriods
  - period
  - periodType


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,pptReplayUrl,assist1PlayerId,...,servedByPlayerId,shootingPlayerId,shotType,winningPlayerId,xCoord,yCoord,zoneCode,maxRegulationPeriods,period,periodType
0,10,00:25,19:35,1551,right,505,goal,11,None,8477015.0,...,None,NaN,tip-in,None,85.0,-1.0,O,3,1,REG
1,12,00:38,19:22,1551,right,507,missed-shot,15,None,NaN,...,None,8479458.0,slap,None,28.0,-37.0,O,3,1,REG
2,15,01:31,18:29,1451,right,506,shot-on-goal,28,None,NaN,...,None,8476853.0,snap,None,-32.0,-2.0,O,3,1,REG
3,50,01:39,18:21,1451,right,508,blocked-shot,32,None,NaN,...,None,8477341.0,None,None,-74.0,7.0,D,3,1,REG
4,18,01:58,18:02,1451,right,507,missed-shot,33,None,NaN,...,None,8478483.0,wrist,None,-46.0,-16.0,O,3,1,REG


## Step 3: Remove Unnecessary Columns


In [5]:
# Remove columns not relevant for shot analysis
df_shots_clean = remove_unnecessary_columns(df_shots_clean)

print(f"Shape after removing columns: {df_shots_clean.shape}")
df_shots_clean.head()


Shape after removing columns: (680581, 28)


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,assist1PlayerId,assist1PlayerTotal,...,homeScore,scoringPlayerId,scoringPlayerTotal,shootingPlayerId,shotType,xCoord,yCoord,zoneCode,period,periodType
0,10,00:25,19:35,1551,right,505,goal,11,8477015.0,1.0,...,0.0,8480801.0,1.0,NaN,tip-in,85.0,-1.0,O,1,REG
1,12,00:38,19:22,1551,right,507,missed-shot,15,NaN,NaN,...,NaN,NaN,NaN,8479458.0,slap,28.0,-37.0,O,1,REG
2,15,01:31,18:29,1451,right,506,shot-on-goal,28,NaN,NaN,...,NaN,NaN,NaN,8476853.0,snap,-32.0,-2.0,O,1,REG
3,50,01:39,18:21,1451,right,508,blocked-shot,32,NaN,NaN,...,NaN,NaN,NaN,8477341.0,None,-74.0,7.0,D,1,REG
4,18,01:58,18:02,1451,right,507,missed-shot,33,NaN,NaN,...,NaN,NaN,NaN,8478483.0,wrist,-46.0,-16.0,O,1,REG


## Step 4: Create Target Variable


In [6]:
# Create is_goal column
df_shots_clean = create_goal_target(df_shots_clean)

print("Goal Counts (1=Goal, 0=No Goal):")
print(df_shots_clean["is_goal"].value_counts())
print(f"\nGoal Rate: {df_shots_clean['is_goal'].mean():.2%}")


Goal Counts (1=Goal, 0=No Goal):
is_goal
0    643641
1     36940
Name: count, dtype: int64

Goal Rate: 5.43%


## Step 5: Standardize Coordinates


In [7]:
# Standardize coordinates so all shots are aimed at positive-x net
df_shots_clean = standardize_coordinates(df_shots_clean)

print("Coordinates standardized. All shots are now aimed at the positive-x net.")
print("\nCoordinate ranges:")
print(f"  xCoord: {df_shots_clean['xCoord'].min():.1f} to {df_shots_clean['xCoord'].max():.1f}")
print(f"  yCoord: {df_shots_clean['yCoord'].min():.1f} to {df_shots_clean['yCoord'].max():.1f}")


Coordinates standardized. All shots are now aimed at the positive-x net.

Coordinate ranges:
  xCoord: 0.0 to 99.0
  yCoord: -42.0 to 42.0


## Step 6: Handle Missing Data


In [8]:
# Check missing data before handling
print("Missing values before handling:")
missing_before = df_shots_clean.isnull().sum()
print(missing_before[missing_before > 0])

# Handle missing data
df_shots_clean = handle_missing_data(df_shots_clean)

# Check missing data after handling
print("\nMissing values after handling:")
missing_after = df_shots_clean.isnull().sum()
print(missing_after[missing_after > 0])

# Check shot type distribution
print("\nShot Type Distribution:")
print(df_shots_clean['shotType'].value_counts())


Missing values before handling:
assist1PlayerId       646964
assist1PlayerTotal    646964
assist2PlayerId       653423
assist2PlayerTotal    653423
awaySOG               350479
awayScore             643641
blockingPlayerId      509744
goalieInNetId         174163
homeSOG               350479
homeScore             643641
scoringPlayerId       643641
scoringPlayerTotal    644618
shootingPlayerId       36940
shotType              170897
xCoord                     1
yCoord                     1
zoneCode                   1
dtype: int64

Missing values after handling:
awaySOG               350479
awayScore             643641
blockingPlayerId      509744
goalieInNetId         174163
homeSOG               350479
homeScore             643641
scoringPlayerId       643641
scoringPlayerTotal    644618
shootingPlayerId       36940
xCoord                     1
yCoord                     1
zoneCode                   1
dtype: int64

Shot Type Distribution:
shotType
wrist           278415
unknown     

## Step 7: Save Cleaned Data


In [9]:
# Save cleaned data
# First, check for and remove any duplicate columns (PyArrow doesn't allow duplicates)
if df_shots_clean.columns.duplicated().any():
    duplicates = df_shots_clean.columns[df_shots_clean.columns.duplicated()].unique()
    print(f"Warning: Found duplicate columns before saving: {list(duplicates)}")
    df_shots_clean = df_shots_clean.loc[:, ~df_shots_clean.columns.duplicated()]
    print(f"Removed duplicates. New shape: {df_shots_clean.shape}")

# Fix data types for PyArrow compatibility
# Convert object columns with mixed types to strings
print("\nFixing data types for PyArrow compatibility...")
for col in df_shots_clean.columns:
    if df_shots_clean[col].dtype == 'object':
        # Check if column has mixed types (numbers and strings)
        try:
            # Try to convert to numeric - if it fails, it's a string column
            pd.to_numeric(df_shots_clean[col], errors='raise')
        except (ValueError, TypeError):
            # It's a string column - ensure all values are strings
            df_shots_clean[col] = df_shots_clean[col].astype(str)
            # Replace 'nan' strings with empty string or 'None'
            df_shots_clean[col] = df_shots_clean[col].replace('nan', 'None')
            df_shots_clean[col] = df_shots_clean[col].replace('NaN', 'None')
            df_shots_clean[col] = df_shots_clean[col].replace('<NA>', 'None')

print("Data types fixed!")

OUTPUT_FILE = "../../data/processed/cleaned_shots.parquet"
df_shots_clean.to_parquet(OUTPUT_FILE, index=False)

print(f"Cleaned data saved to {OUTPUT_FILE}")
print(f"\nFinal DataFrame shape: {df_shots_clean.shape}")
print(f"Total shots: {len(df_shots_clean):,}")
print(f"Total goals: {df_shots_clean['is_goal'].sum():,}")
print(f"Goal rate: {df_shots_clean['is_goal'].mean():.2%}")

df_shots_clean.head()



Fixing data types for PyArrow compatibility...
Data types fixed!
Cleaned data saved to ../../data/processed/cleaned_shots.parquet

Final DataFrame shape: (680581, 29)
Total shots: 680,581
Total goals: 36,940
Goal rate: 5.43%


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,assist1PlayerId,assist1PlayerTotal,...,scoringPlayerId,scoringPlayerTotal,shootingPlayerId,shotType,xCoord,yCoord,zoneCode,period,periodType,is_goal
0,10,00:25,19:35,1551,right,505,goal,11,8477015.0,1.0,...,8480801.0,1.0,NaN,tip-in,85.0,-1.0,O,1,REG,1
1,12,00:38,19:22,1551,right,507,missed-shot,15,None,None,...,NaN,NaN,8479458.0,slap,28.0,-37.0,O,1,REG,0
2,15,01:31,18:29,1451,right,506,shot-on-goal,28,None,None,...,NaN,NaN,8476853.0,snap,32.0,2.0,O,1,REG,0
3,50,01:39,18:21,1451,right,508,blocked-shot,32,None,None,...,NaN,NaN,8477341.0,unknown,74.0,-7.0,D,1,REG,0
4,18,01:58,18:02,1451,right,507,missed-shot,33,None,None,...,NaN,NaN,8478483.0,wrist,46.0,16.0,O,1,REG,0
